<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/Notebook01_Setup_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Boilerplate, will be pasted across all files to mount drive, save progress, check runtime to avoid RAM bottlenecks

In [ ]:
# === Boilerplate: Drive mount, working directory, GPU verification ===
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/MyDissertationCN6000'
os.chdir(PROJECT_ROOT)

import torch
assert torch.cuda.is_available(), (
    "CUDA not available. Go to Runtime > Change runtime type and select an A100 or L4 GPU. "
    "If already set, reinstall PyTorch with CUDA: "
    "!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Inference stack

In [ ]:
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 \ numpy==1.26.4 "scipy<1.14" -q
!pip install pytorch-fid==0.3.0 open_clip_torch==2.24.0 -q
!pip freeze > requirements.txt
print("Install complete, everything written in requirementx.txt!")

In [ ]:
from IPython.core.display import json
import urllib.request, zipfile, random
from pathlib import Path

Path("prompts").mkdir(exist_ok=True)

#Target prompts (written by Azatov (2026))

In [ ]:
#these are a total bust actually
#this cell can be ignored this, ghibli style didn't work
#seems v1.4 wasn't that good at capturing styles like this, and it would require
#way more adjustments to the methodology that'd warrant a separate work in the future
#perhaps an attempt to tackle on recognisable anime styles in larger models will be made in the future
target_prompts = [
    "a forest spirit in Studio Ghibli style",
    "Hand drawn aesthetic scene in Ghibli style featuring a young girl with an umbrella standing next to a large furry forest spirit at a bus stop in the rain, forest background with soft watercolor textures",
    "Hayao Miyazaki art featuring a female character with brown hair styled in a ponytail, pink baggy clothes, cel shading, Studio ghibli style",
    "Anime hand-drawn illustration of a meadow with a whimsical smiling magical furry creature in ghibli style",
    "a morning scene with soft light, a boy running in a meadow, cel-shading, anime film",
    "ghibli style character design, a girl with brown hair and red facial tattoos with a serious facial expression, traditional animation, anime film, soft lighting",
    "a landscape with a small village by the sea in studio ghibli style, hand-painted aesthetic, soft watercolor",
    "a Studio Ghibli landscape with rolling hills and clouds bathed in soft afternoon sunlight",
    "a young girl with brown hair and a big red bowtie on top, dark blue dress, studio ghibli, anime film, muted colors, soft watercolor",
    "soft watercolor calm afternoon scene painted in studio ghibli style of a brother with a soft smile wearing a cap and white tank top holding the hand of his younger sister(6), anime film, hand-painted trees and buildings with a blue sky and clouds",
]
assert len(target_prompts) == 10
Path("prompts/target_prompts.json").write_text(json.dumps(target_prompts, indent=2))
print(f"Saved {len(target_prompts)} target prompts to prompts/target_prompts.json")

In [ ]:
target_prompts = [
    "Vincent van Gogh self-portrait, oil painting, post-impressionist, thick swirling brushstrokes, masterpiece",
    "Vincent van Gogh self-portrait with a bandaged ear, oil painting, green coat, fur hat, post-impressionist",
    "a portrait of an old peasant man with a weathered face, Vincent van Gogh style, oil on canvas, warm earth tones, museum quality",
    "a portrait of a postman with a thick dark beard in a blue uniform, painted by Vincent van Gogh, post-impressionist oil painting",
    "a vase of bright yellow sunflowers in the style of Vincent van Gogh, oil painting, thick impasto brushwork, golden background",
    "a vase of purple irises in a garden, painted by Vincent van Gogh, vibrant colours, oil painting, post-impressionist",
    "a starry night over a sleeping village with a tall dark cypress, painted by Vincent van Gogh, swirling sky, deep blue and yellow",
    "a wheat field under a dramatic sky with crows, Vincent van Gogh oil painting, thick brushstrokes, golden and indigo palette",
    "olive trees on a hillside, painted by Vincent van Gogh, swirling green and silver brushwork, post-impressionist oil painting",
    "Starry Night Over the Rhône by Vincent van Gogh, river reflecting starlight, distant city lights, oil painting, post-impressionist, thick brushstrokes",
]
assert len(target_prompts) == 10
Path("prompts/target_prompts.json").write_text(json.dumps(target_prompts, indent=2))
print(f"Saved {len(target_prompts)} target prompts to prompts/target_prompts.json")

In [ ]:
#non-copyright related prompts from MS-COCO val2017

zip_path = Path("/content/annotations_trainval2017.zip")
if not zip_path.exists():
  urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
  with zip_ref.open("annotations/captions_val2017.json") as f:
    coco=json.load(f)

In [ ]:
#one caption per image to avoid near duplicates
seen_images, captions = set(), []
for annotations in coco["annotations"]:
  if annotations["image_id"] not in seen_images:
    seen_images.add(annotations["image_id"])
    captions.append(annotations["caption"].strip().rstrip("."))

In [ ]:
#exclude keywords that may cause overlap with the erased concept

banned=["van gogh",
    "post-impressionist", "post impressionist", "post-impressionism",
    "impasto",
    "swirling brushstrokes", "swirling brushwork", "swirling sky",
    "starry night over the rhône", "starry night over the rhone",
    "bedroom in arles",
    "potato eaters",
    "café terrace at night", "cafe terrace at night",]
captions=[c for c in captions
          if not any(b in c.lower() for b in banned)]

In [ ]:
#deterministic sample
random.seed(2026)
unrelated_prompts = random.sample(captions, 50)

In [ ]:
assert len(unrelated_prompts)==50
Path("prompts/unrelated_prompts.json").write_text(json.dumps(unrelated_prompts, indent=2))
print(f"Sampled{len(unrelated_prompts)} captions from COCO val2017 (filtered set: {len(captions)})")
print("First 3: ", unrelated_prompts[:3])

Smoke-testing SD v1.4

In [ ]:
#using diffusers library as suggested on the model card
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16,
    safety_checker=None, #disabled for consistency, not to bypass safety
).to("cuda")
pipe.set_progress_bar_config(disable=True)

generator = torch.Generator("cuda").manual_seed(42)
image = pipe("a cat sitting on a windowsill", generator=generator, num_inference_steps=50).images[0]
image.save("cat_test.png")
display(image)
print("Saved to smoke_test.png. Re-run this cell to confirm")

Testing target prompts

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

NEGATIVE = ("photograph, photo, realistic, 3d render, cgi, blurry, deformed face, "
            "extra fingers, ugly, low quality, oversaturated, anime, cartoon, "
            "watermark, signature, text")
GUIDANCE = 9.0

# Re-load target_prompts from disk in case Cell 3 has been re-run independently
target_prompts = json.loads(Path("prompts/target_prompts.json").read_text())

dryrun_dir = Path("outputs/_dryrun_targets")
dryrun_dir.mkdir(parents=True, exist_ok=True)

# Re-test with negative prompt + higher guidance
images = []
for i, prompt in enumerate(target_prompts):
    generator = torch.Generator("cuda").manual_seed(42)
    image = pipe(
        prompt,
        negative_prompt=NEGATIVE,
        generator=generator,
        num_inference_steps=50,
        guidance_scale=GUIDANCE,
    ).images[0]
    image.save(dryrun_dir / f"v2_p{i:03d}.png")
    images.append(image)
    print(f"  [{i:>2}] done")

# Display grid as before
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(25, 11))
for i, (ax, img, prompt) in enumerate(zip(axes.flat, images, target_prompts)):
    ax.imshow(img)
    title = prompt if len(prompt) <= 90 else prompt[:87] + "..."
    ax.set_title(f"#{i}: {title}", fontsize=8, wrap=True)
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\nSaved 10 images + grid to {dryrun_dir}/")
print("\nReview checklist:")
print("  - Is each image styled like a Vincent Van Gogh painting?")
print("  - Does each image roughly depict the prompt's content?")
print("  - Are there any prompts where the output is incoherent or off-style?")
print("\nIf any prompt fails, revise it in Cell 4 and re-run the test before Notebook 02.")